# Epilepsy & Psychiatric Comorbidities — Cohort Building & STROBE Flowchart

**Project:** Prevalence and Trends of DSM-5 Axis I Psychiatric Disorders in Epilepsy Surgery  
**Data:** NIS 2011-2020 (excluding 2015 transition year)  
**Cohort:** Extracted via `extract_epilepsy.py` → `epilepsy_cohort.parquet`

This notebook:
1. Loads the extracted epilepsy cohort
2. Applies inclusion/exclusion criteria (adults ≥18, valid survey vars)
3. Reports STROBE flowchart numbers at each step
4. Validates demographics, surgery rates, and psychiatric prevalence
5. Saves the analytic cohort

In [ ]:
# ============================================================
# SETUP
# ============================================================
library(arrow)
library(data.table)

datadir <- "/Volumes/Niels 2/NIS_new_version/NIS_epilepsy_psych/output"
outdir  <- "/Volumes/Niels 2/NIS_new_version/NIS_epilepsy_psych"

cat("Loading epilepsy cohort from Parquet...\n")
dt <- as.data.table(read_parquet(file.path(datadir, "epilepsy_cohort.parquet")))
cat(sprintf("Loaded: %s rows x %d columns\n", format(nrow(dt), big.mark = ","), ncol(dt)))
cat(sprintf("Years: %s\n", paste(sort(unique(dt$YEAR)), collapse = ", ")))
invisible(NULL)

## STROBE Flowchart — Step-by-Step Filtering

In [ ]:
# ============================================================
# STROBE FLOWCHART NUMBERS
# ============================================================
flow <- list()

# Total NIS (from extraction log)
flow$total_nis <- 73397393
flow$epilepsy_dx1_raw <- nrow(dt)

cat("=" |> rep(60) |> paste(collapse = ""), "\n")
cat("STROBE FLOWCHART\n")
cat("=" |> rep(60) |> paste(collapse = ""), "\n\n")

cat(sprintf("STEP 0: Total NIS 2011-2020: %s hospitalizations\n", format(flow$total_nis, big.mark = ",")))
cat(sprintf("STEP 1: Epilepsy as DX1 (345.xx / G40.x), excl 2015: %s\n\n", 
            format(flow$epilepsy_dx1_raw, big.mark = ",")))

# By ICD version
cat("By ICD version:\n")
cat(sprintf("  ICD-9 (2011-2014):  %s\n", format(sum(dt$icd_version == 9), big.mark = ",")))
cat(sprintf("  ICD-10 (2016-2020): %s\n", format(sum(dt$icd_version == 10), big.mark = ",")))

cat("\nBy year:\n")
print(table(dt$YEAR))
invisible(NULL)

In [ ]:
# ============================================================
# STEP 2: Exclude pediatric (age < 18)
# ============================================================
flow$pediatric <- sum(dt$AGE < 18, na.rm = TRUE)
dt <- dt[AGE >= 18 | is.na(AGE)]
flow$after_age <- nrow(dt)

cat(sprintf("STEP 2: Exclude age < 18\n"))
cat(sprintf("  Excluded (pediatric): %s\n", format(flow$pediatric, big.mark = ",")))
cat(sprintf("  Remaining: %s\n\n", format(flow$after_age, big.mark = ",")))

# Age summary
cat("Age summary:\n")
print(summary(dt$AGE))
invisible(NULL)

In [ ]:
# ============================================================
# STEP 3: Exclude missing survey variables
# ============================================================
missing_survey <- is.na(dt$HOSP_NIS) | is.na(dt$NIS_STRATUM) | is.na(dt$DISCWT)
flow$missing_survey <- sum(missing_survey)
dt <- dt[!missing_survey]
flow$after_survey <- nrow(dt)

cat(sprintf("STEP 3: Exclude missing survey variables (HOSP_NIS/NIS_STRATUM/DISCWT)\n"))
cat(sprintf("  Excluded: %s\n", format(flow$missing_survey, big.mark = ",")))
cat(sprintf("  Remaining (ANALYTIC COHORT): %s\n", format(flow$after_survey, big.mark = ",")))
invisible(NULL)

## Cohort Characterization

In [ ]:
# ============================================================
# EPILEPSY TYPE & INTRACTABILITY
# ============================================================
cat("Epilepsy type classification:\n")
type_tab <- table(dt$epilepsy_type)
for (t in sort(names(type_tab))) {
    cat(sprintf("  %-20s %s (%5.1f%%)\n", t, format(type_tab[t], big.mark = ","),
                100 * type_tab[t] / nrow(dt)))
}

cat(sprintf("\nIntractable epilepsy: %s (%4.1f%%)\n",
            format(sum(dt$intractable), big.mark = ","),
            100 * mean(dt$intractable)))

cat("\nBy year:\n")
print(table(dt$YEAR, dt$epilepsy_type))
invisible(NULL)

In [ ]:
# ============================================================
# SURGERY GROUPS
# ============================================================
cat("Surgery group breakdown:\n\n")
surg_tab <- table(dt$surgery_group)
for (g in c("Resective", "VNS", "RNS", "Monitoring", "None")) {
    n <- ifelse(g %in% names(surg_tab), surg_tab[g], 0)
    cat(sprintf("  %-15s %s (%5.2f%%)\n", g, format(n, big.mark = ","),
                100 * n / nrow(dt)))
}

cat(sprintf("\nAny surgery: %s (%4.1f%%)\n",
            format(sum(dt$any_surgery), big.mark = ","),
            100 * mean(dt$any_surgery)))

# Surgery by year
cat("\nSurgery cases by year:\n")
surg_year <- dt[any_surgery == TRUE, .N, by = YEAR][order(YEAR)]
total_year <- dt[, .N, by = YEAR][order(YEAR)]
merged <- merge(total_year, surg_year, by = "YEAR", suffixes = c("_total", "_surg"))
merged[, pct := round(100 * N_surg / N_total, 2)]
print(merged)

# Surgery by type
cat("\nSurgery type by epilepsy type:\n")
print(table(dt[any_surgery == TRUE]$epilepsy_type, dt[any_surgery == TRUE]$surgery_group))
invisible(NULL)

In [ ]:
# ============================================================
# PSYCHIATRIC COMORBIDITY OVERVIEW
# ============================================================
psych_cols <- grep("^psych_", names(dt), value = TRUE)

cat("Psychiatric comorbidity prevalence (unweighted):\n\n")
cat(sprintf("  %-25s %8s  %6s\n", "Category", "N", "%"))
cat(paste(rep("-", 45), collapse = ""), "\n")
for (col in c(psych_cols, "any_psych")) {
    n <- sum(dt[[col]], na.rm = TRUE)
    label <- gsub("^psych_", "", col)
    if (col == "any_psych") label <- "ANY PSYCHIATRIC"
    cat(sprintf("  %-25s %8s  %5.1f%%\n", label, format(n, big.mark = ","),
                100 * n / nrow(dt)))
}
invisible(NULL)

In [ ]:
# ============================================================
# PSYCHIATRIC PREVALENCE BY SURGERY STATUS
# ============================================================
cat("Psychiatric comorbidity by surgery status:\n\n")

for (grp_label in c("Any Surgery", "Resective", "VNS", "RNS", "Monitoring", "No Surgery")) {
    if (grp_label == "Any Surgery") {
        sub <- dt[any_surgery == TRUE]
    } else if (grp_label == "No Surgery") {
        sub <- dt[any_surgery == FALSE]
    } else {
        sub <- dt[surgery_group == grp_label]
    }
    cat(sprintf("--- %s (N=%s) ---\n", grp_label, format(nrow(sub), big.mark = ",")))
    for (col in c(psych_cols, "any_psych")) {
        n <- sum(sub[[col]], na.rm = TRUE)
        label <- gsub("^psych_", "", col)
        if (col == "any_psych") label <- "ANY PSYCHIATRIC"
        cat(sprintf("  %-25s %6s (%5.1f%%)\n", label, format(n, big.mark = ","),
                    100 * n / nrow(sub)))
    }
    cat("\n")
}
invisible(NULL)

## Demographics

In [ ]:
# ============================================================
# DEMOGRAPHICS
# ============================================================
cat("DEMOGRAPHICS\n\n")

cat("Age:\n")
cat(sprintf("  Mean: %.1f, Median: %.0f, SD: %.1f\n",
            mean(dt$AGE, na.rm = TRUE), median(dt$AGE, na.rm = TRUE),
            sd(dt$AGE, na.rm = TRUE)))

cat("\nSex (Female):\n")
cat(sprintf("  %s / %s (%.1f%%)\n",
            format(sum(dt$FEMALE == 1, na.rm = TRUE), big.mark = ","),
            format(nrow(dt), big.mark = ","),
            100 * mean(dt$FEMALE == 1, na.rm = TRUE)))

cat("\nRace:\n")
race_labels <- c("1" = "White", "2" = "Black", "3" = "Hispanic",
                 "4" = "Asian/PI", "5" = "Native Am", "6" = "Other")
race_tab <- table(dt$RACE, useNA = "ifany")
for (r in names(race_tab)) {
    lbl <- ifelse(r %in% names(race_labels), race_labels[r], paste0("Code ", r))
    cat(sprintf("  %-15s %s (%.1f%%)\n", lbl, format(race_tab[r], big.mark = ","),
                100 * race_tab[r] / nrow(dt)))
}

cat("\nPayer:\n")
pay_labels <- c("1" = "Medicare", "2" = "Medicaid", "3" = "Private",
                "4" = "Self-pay", "5" = "No charge", "6" = "Other")
pay_tab <- table(dt$PAY1, useNA = "ifany")
for (p in names(pay_tab)) {
    lbl <- ifelse(p %in% names(pay_labels), pay_labels[p], paste0("Code ", p))
    cat(sprintf("  %-15s %s (%.1f%%)\n", lbl, format(pay_tab[p], big.mark = ","),
                100 * pay_tab[p] / nrow(dt)))
}

cat("\nIncome quartile:\n")
print(table(dt$ZIPINC_QRTL, useNA = "ifany"))

cat("\nOutcomes:\n")
cat(sprintf("  Mortality: %s (%.1f%%)\n",
            format(sum(dt$DIED == 1, na.rm = TRUE), big.mark = ","),
            100 * mean(dt$DIED == 1, na.rm = TRUE)))
cat(sprintf("  LOS mean: %.1f (SD %.1f), median: %.0f\n",
            mean(dt$LOS, na.rm = TRUE), sd(dt$LOS, na.rm = TRUE),
            median(dt$LOS, na.rm = TRUE)))
if ("TOTCHG" %in% names(dt)) {
    cat(sprintf("  Total charges mean: $%s\n",
                format(round(mean(dt$TOTCHG, na.rm = TRUE)), big.mark = ",")))
}
invisible(NULL)

## Validation Checks

In [ ]:
# ============================================================
# VALIDATION: Compare to literature
# ============================================================
cat("VALIDATION CHECKS\n\n")

# 1. Weighted national estimate
weighted_total <- sum(dt$DISCWT, na.rm = TRUE)
cat(sprintf("1. Weighted national epilepsy DX1 estimate: %s\n",
            format(round(weighted_total), big.mark = ",")))
cat("   (Literature: ~150,000-200,000 epilepsy hospitalizations/year nationally)\n")
cat(sprintf("   Per-year weighted avg: %s\n\n",
            format(round(weighted_total / length(unique(dt$YEAR))), big.mark = ",")))

# 2. Surgery rate
cat(sprintf("2. Surgery rate: %.2f%% of epilepsy hospitalizations\n",
            100 * mean(dt$any_surgery)))
cat("   Resective surgery rate among epilepsy admissions (literature: ~0.5-1%%)\n\n")

# 3. Psychiatric prevalence
cat(sprintf("3. Any psychiatric comorbidity: %.1f%%\n", 100 * mean(dt$any_psych)))
cat("   Literature (Kaushal 2017 NIS): depression 13%%, psychosis 10.4%%\n")
cat(sprintf("   Our depression: %.1f%%, schizophrenia: %.1f%%, psychosis: %.1f%%\n\n",
            100 * mean(dt$psych_depression),
            100 * mean(dt$psych_schizophrenia),
            100 * mean(dt$psych_psychosis)))

# 4. Demographics by surgery status
cat("4. Demographics by surgery status:\n")
for (grp in c(TRUE, FALSE)) {
    sub <- dt[any_surgery == grp]
    lbl <- ifelse(grp, "Surgery", "No Surgery")
    cat(sprintf("  %s (N=%s): Age %.1f, Female %.1f%%, Mortality %.1f%%, LOS %.1f\n",
                lbl, format(nrow(sub), big.mark = ","),
                mean(sub$AGE, na.rm = TRUE),
                100 * mean(sub$FEMALE == 1, na.rm = TRUE),
                100 * mean(sub$DIED == 1, na.rm = TRUE),
                mean(sub$LOS, na.rm = TRUE)))
}
invisible(NULL)

## STROBE Flowchart Summary

In [ ]:
# ============================================================
# STROBE FLOWCHART SUMMARY
# ============================================================
cat("============================================================\n")
cat("STROBE FLOWCHART SUMMARY\n")
cat("============================================================\n\n")

cat(sprintf("Total NIS 2011-2020:                    %s\n", format(flow$total_nis, big.mark = ",")))
cat(sprintf("  Non-epilepsy excluded:               -%s\n", format(flow$total_nis - flow$epilepsy_dx1_raw, big.mark = ",")))
cat(sprintf("  2015 transition year excluded:        (excluded in extraction)\n"))
cat(sprintf("\nEpilepsy DX1 (excl 2015):               %s\n", format(flow$epilepsy_dx1_raw, big.mark = ",")))
cat(sprintf("  Excluded age < 18:                   -%s\n", format(flow$pediatric, big.mark = ",")))
cat(sprintf("\nAdult epilepsy:                         %s\n", format(flow$after_age, big.mark = ",")))
cat(sprintf("  Excluded missing survey vars:         -%s\n", format(flow$missing_survey, big.mark = ",")))
cat(sprintf("\n*** ANALYTIC COHORT:                    %s ***\n", format(flow$after_survey, big.mark = ",")))

cat(sprintf("\n  Epilepsy surgery:      %s (%.1f%%)\n",
            format(sum(dt$any_surgery), big.mark = ","),
            100 * mean(dt$any_surgery)))
cat(sprintf("    Resective:           %s\n", format(sum(dt$surgery_group == "Resective"), big.mark = ",")))
cat(sprintf("    VNS:                 %s\n", format(sum(dt$surgery_group == "VNS"), big.mark = ",")))
cat(sprintf("    RNS:                 %s\n", format(sum(dt$surgery_group == "RNS"), big.mark = ",")))
cat(sprintf("    Monitoring:          %s\n", format(sum(dt$surgery_group == "Monitoring"), big.mark = ",")))
cat(sprintf("  Non-surgical epilepsy: %s (%.1f%%)\n",
            format(sum(!dt$any_surgery), big.mark = ","),
            100 * mean(!dt$any_surgery)))

# Save flowchart numbers
flow_df <- data.frame(
    step = c("total_nis", "epilepsy_dx1_raw", "pediatric_excluded",
             "adult_epilepsy", "missing_survey", "analytic_cohort",
             "any_surgery", "resective", "vns", "rns", "monitoring", "no_surgery"),
    value = c(flow$total_nis, flow$epilepsy_dx1_raw, flow$pediatric,
              flow$after_age, flow$missing_survey, flow$after_survey,
              sum(dt$any_surgery), sum(dt$surgery_group == "Resective"),
              sum(dt$surgery_group == "VNS"), sum(dt$surgery_group == "RNS"),
              sum(dt$surgery_group == "Monitoring"), sum(!dt$any_surgery))
)
write.csv(flow_df, file.path(outdir, "tables/strobe_flowchart_numbers.csv"), row.names = FALSE)
cat("\nSaved flowchart numbers to tables/strobe_flowchart_numbers.csv\n")
invisible(NULL)

## Save Analytic Cohort

In [ ]:
# ============================================================
# SAVE ANALYTIC COHORT
# ============================================================
analytic_path <- file.path(outdir, "output/epilepsy_analytic.parquet")
write_parquet(dt, analytic_path)
cat(sprintf("Saved analytic cohort: %s\n", analytic_path))
cat(sprintf("  Rows: %s\n", format(nrow(dt), big.mark = ",")))
cat(sprintf("  Cols: %d\n", ncol(dt)))
cat(sprintf("  File size: %.1f MB\n", file.size(analytic_path) / 1e6))
cat("\nDONE. Proceed to 02_prevalence.ipynb\n")
invisible(NULL)